# 04 · Promotion Lift Testing

Matched within-store design: each promo day is compared to the same store and weekday at t−7 and t+7 (non-promo), excluding state holidays and December. Promo is chain-wide (every date is 0% or 100% of stores) and weekday-only, so there is no same-day control group.

## 1. Load data

In [ ]:
import pandas as pd, numpy as np, pyodbc
import matplotlib.pyplot as plt

conn = pyodbc.connect("Driver={ODBC Driver 18 for SQL Server};Server=localhost;"
                      "Database=RetailForecast;Trusted_Connection=yes;TrustServerCertificate=yes;")
df = pd.read_sql("""
    SELECT d.Store, d.SalesDate AS [Date], d.DayOfWeek, d.Sales, d.Customers, d.Promo,
           d.StateHoliday, d.SchoolHoliday, c.Cluster
    FROM daily_sales d JOIN store_clusters c ON c.Store = d.Store
    WHERE d.IsOpen = 1 AND d.Sales > 0""", conn, parse_dates=['Date'])
df['Promo'] = df['Promo'].astype(int)
df['Hol'] = (df['StateHoliday'].astype(str).str.strip() != '0').astype(int)
print(df.shape)

## 2. Is Promo chain-wide?

In [ ]:
share = df.groupby('Date')['Promo'].mean()
print(pd.cut(share, [-.01, .01, .99, 1]).value_counts())
print(df.groupby('DayOfWeek')['Promo'].mean().round(2))

## 3. Matched design (same store, same weekday, ±7 days)

In [ ]:
cols = ['Sales', 'Customers', 'Promo', 'Hol']

def shifted(days, sfx):
    x = df[['Store', 'Date'] + cols].copy()
    x['Date'] += pd.Timedelta(days=days)   # +7 brings t-7 onto t
    return x.rename(columns={c: c + sfx for c in cols})

m = (df.merge(shifted(7, '_prev'), on=['Store', 'Date'])
       .merge(shifted(-7, '_next'), on=['Store', 'Date']))
m = m[(m.Promo == 1) & (m.Promo_prev == 0) & (m.Promo_next == 0)
      & (m[['Hol', 'Hol_prev', 'Hol_next']].sum(axis=1) == 0)
      & (m.Date.dt.month != 12)].copy()

for v in ['Sales', 'Customers']:
    m[f'{v}_ctrl'] = (m[f'{v}_prev'] + m[f'{v}_next']) / 2

print(f"Matched pairs: {len(m):,} | stores: {m.Store.nunique()}")

## 4. Overall lift, confidence interval, decomposition

In [ ]:
def lift(g, v='Sales'):
    return g[v].sum() / g[f'{v}_ctrl'].sum() - 1

def summarize(g):
    s, c = lift(g), lift(g, 'Customers')
    return pd.Series({'sales_lift': s, 'traffic_lift': c,
                      'basket_lift': (1 + s) / (1 + c) - 1, 'stores': g.Store.nunique()})

rng = np.random.default_rng(42)
by_store = m.groupby('Store')[['Sales', 'Sales_ctrl']].sum()
boots = [(lambda s: s.Sales.sum() / s.Sales_ctrl.sum() - 1)(
         by_store.sample(len(by_store), replace=True, random_state=rng)) for _ in range(1000)]

overall = summarize(m)
print(f"Sales lift   {overall.sales_lift:.1%}  (95% CI {np.percentile(boots, 2.5):.1%} – {np.percentile(boots, 97.5):.1%})")
print(f"Traffic lift {overall.traffic_lift:.1%} | Basket lift {overall.basket_lift:.1%}")

dow = m.groupby('DayOfWeek')[m.columns].apply(summarize)
print(dow[['sales_lift', 'traffic_lift', 'basket_lift']].map('{:.1%}'.format))

## 5. Lift by cluster

Cluster labels: 1 weekday/promo-sensitive, 2 summer/tourist, 3 Christmas-driven, 4 Sunday-trading, 5 balanced week.

In [ ]:
names = {1: 'Weekday/promo', 2: 'Summer/tourist', 3: 'Christmas', 4: 'Sunday-trading', 5: 'Balanced'}
clu = m.groupby('Cluster')[m.columns].apply(summarize)
clu.index = clu.index.map(lambda k: f"{k} {names.get(k, '')}")
out = clu.copy()
for c in ['sales_lift', 'traffic_lift', 'basket_lift']:
    out[c] = out[c].map('{:.1%}'.format)
out['stores'] = out['stores'].astype(int)
print(out)

## 6. Store-level lift distribution

In [ ]:
store_lift = (m.groupby(['Store', 'Cluster'])[m.columns].apply(summarize)
                .reset_index().drop(columns='stores'))
print(store_lift['sales_lift'].describe(percentiles=[.05, .25, .5, .75, .95]).map('{:.3f}'.format))
print("\nLowest 5:\n", store_lift.nsmallest(5, 'sales_lift').round(3).to_string(index=False))
print("\nHighest 5:\n", store_lift.nlargest(5, 'sales_lift').round(3).to_string(index=False))
print("\nSample stores:\n", store_lift[store_lift.Store.isin([314, 299, 763, 259, 640])].round(3).to_string(index=False))

## 6b. Anomaly check: stores with ~0% lift (789, 794)

In [ ]:
for s in [789, 794]:
    d = df[df.Store == s]
    print(f"Store {s}: {d.Date.min().date()} to {d.Date.max().date()} | {len(d)} open days | "
          f"matched pairs {len(m[m.Store == s])}")
    print(d.groupby(['DayOfWeek', 'Promo'])['Sales'].mean().unstack().round(0), "\n")

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for a, s in zip(ax, [789, 794]):
    d = df[df.Store == s].set_index('Date')
    a.plot(d['Sales'], lw=0.6, color='grey')
    a.scatter(d.index[d.Promo == 1], d.loc[d.Promo == 1, 'Sales'], s=4, color='#DD8452', label='Promo day')
    a.set(title=f'Store {s} daily sales', ylabel='Sales')
    a.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 7. Figures

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Traffic and basket lifts multiply, so split the total sales lift by log share
t_share = np.log1p(dow['traffic_lift']) / np.log1p(dow['sales_lift'])
total = dow['sales_lift'] * 100
d = pd.DataFrame({'Traffic': total * t_share, 'Basket': total * (1 - t_share)})
d.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri'][:len(d)]
d.plot(kind='bar', stacked=True, ax=ax[0], rot=0, color=['#4C72B0', '#DD8452'])
for i, v in enumerate(total):
    ax[0].text(i, v + 1.5, f"{v:.0f}%", ha='center')
ax[0].set_ylim(0, total.max() * 1.15)
ax[0].set(title='Promo sales lift by weekday\n(split into traffic vs basket contribution)', ylabel='Sales lift (%)', xlabel='')

ax[1].hist(store_lift['sales_lift'] * 100, bins=50, color='#4C72B0')
ax[1].axvline(overall.sales_lift * 100, color='k', ls='--', label=f"Chain {overall.sales_lift:.1%}")
ax[1].set(title=f'Store-level promo sales lift (n = {len(store_lift):,})', xlabel='Lift (%)', ylabel='Stores')
ax[1].annotate('789, 794\n(non-responders)', xy=(0, 2), xytext=(8, 35),
               arrowprops=dict(arrowstyle='->', color='grey'), fontsize=9)
ax[1].legend()

plt.tight_layout()
plt.savefig('../figures/promo_lift_weekday_store.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save results

In [ ]:
store_lift.to_csv('../data/processed/promo_lift_by_store.csv', index=False)
print("Saved", len(store_lift), "stores")

## 9. Post-promo stockpiling test (6c)

The promo calendar alternates almost perfectly, so the ±7-day controls in section 3 are weeks right after a promo. If shoppers stockpile, those controls are depressed and the +37% is overstated.

A *clean* baseline (non-promo week after a non-promo week) barely exists, so instead we use **back-to-back promo weeks**: promo days in a promo week that follows another promo week vs promo days in a promo week that follows a normal week (same store, same weekday, within ±4 weeks). A negative gap = stockpiling.

In [ ]:
df['Week'] = df['Date'] - pd.to_timedelta(df['Date'].dt.dayofweek, unit='D')   # Monday of week
wk = df.groupby('Week')['Promo'].max().to_frame('promo').asfreq('W-MON')
wk['prev_promo'] = wk['promo'].shift(1)
wk['cls'] = np.select([(wk.promo == 1) & (wk.prev_promo == 1),
                       (wk.promo == 1) & (wk.prev_promo == 0),
                       (wk.promo == 0) & (wk.prev_promo == 1),
                       (wk.promo == 0) & (wk.prev_promo == 0)],
                      ['promo_after_promo', 'promo_after_off', 'off_after_promo', 'off_after_off'], 'unknown')
print(wk['cls'].value_counts())
print("\nClean non-promo weeks (off after off):", [str(x.date()) for x in wk.index[wk.cls == 'off_after_off']])
print("Back-to-back promo weeks:", [str(x.date()) for x in wk.index[wk.cls == 'promo_after_promo']])

In [ ]:
# Promo days only; drop disrupted store-weeks, holidays and the Christmas run-up (Dec 11–31)
d = df.merge(wk[['cls']], left_on='Week', right_index=True)
open_ms = d[d.DayOfWeek <= 6].groupby(['Store', 'Week']).size().rename('open_ms')
d = d.merge(open_ms, left_on=['Store', 'Week'], right_index=True)
xmas = (d.Date.dt.month == 12) & (d.Date.dt.day >= 11)
ok = d[(d.open_ms >= 6) & ~xmas & (d.Hol == 0)]

pp = ok.loc[(ok.cls == 'promo_after_promo') & (ok.Promo == 1), ['Store', 'Date', 'DayOfWeek', 'Cluster', 'Sales', 'Customers']]
ref = ok.loc[(ok.cls == 'promo_after_off') & (ok.Promo == 1), ['Store', 'Date', 'Sales', 'Customers']]

pairs = []
for k in [-28, -21, -14, -7, 7, 14, 21, 28]:
    a = ref.copy()
    a['Date'] -= pd.Timedelta(days=k)          # reference day at t+k aligned onto t
    pairs.append(pp[['Store', 'Date']].merge(a, on=['Store', 'Date']))
refm = (pd.concat(pairs).groupby(['Store', 'Date'])
          .agg(Sales_ref=('Sales', 'mean'), Customers_ref=('Customers', 'mean')))
bb = pp.merge(refm, on=['Store', 'Date'])
print(f"Back-to-back promo days matched: {len(bb):,} | stores: {bb.Store.nunique()}")

In [ ]:
def gap(g, v='Sales'):
    return g[v].sum() / g[f'{v}_ref'].sum() - 1

if len(bb):
    by_s = bb.groupby('Store')[['Sales', 'Sales_ref']].sum()
    rng = np.random.default_rng(42)
    bboots = [(lambda s: s.Sales.sum() / s.Sales_ref.sum() - 1)(
              by_s.sample(len(by_s), replace=True, random_state=rng)) for _ in range(1000)]
    eff, lo, hi = gap(bb), np.percentile(bboots, 2.5), np.percentile(bboots, 97.5)
    print(f"2nd consecutive promo week vs promo week after a normal week: "
          f"sales {eff:+.1%} (95% CI {lo:+.1%} to {hi:+.1%}), traffic {gap(bb, 'Customers'):+.1%}")
    print(bb.groupby('DayOfWeek')[bb.columns].apply(gap).map('{:+.1%}'.format).rename('gap'))
    print(bb.groupby('Cluster')[bb.columns].apply(gap).map('{:+.1%}'.format).rename('gap'))
else:
    eff, lo, hi = np.nan, np.nan, np.nan
    print("No usable back-to-back promo weeks after filters.")

stockpiling = bool(len(bb) and hi < 0)
print("\nStockpiling detected" if stockpiling else "\nNo significant stockpiling detected")

## 10. Corrected lift and Saturday effect

Section 3 controls (and non-promo Saturdays) sit in weeks right after a promo. If stockpiling is significant, the lift vs a normal baseline = (1 + matched lift) × (1 + gap) − 1. Otherwise the matched lift stands.

In [ ]:
corrected = (1 + overall.sales_lift) * (1 + eff) - 1 if stockpiling else overall.sales_lift
print(f"Matched lift {overall.sales_lift:.1%} → corrected {corrected:.1%}")

# Saturday of promo weeks (promo never runs Saturday) vs the same store's Saturdays ±7 days
sat = ok.loc[ok.DayOfWeek == 6, ['Store', 'Date', 'Cluster', 'cls', 'Sales', 'Customers']]
sp = sat.copy()
for k, sfx in [(7, '_prev'), (-7, '_next')]:
    x = sat[['Store', 'Date', 'cls', 'Sales', 'Customers']].copy()
    x['Date'] += pd.Timedelta(days=k)
    sp = sp.merge(x.rename(columns={col: col + sfx for col in ['cls', 'Sales', 'Customers']}),
                  on=['Store', 'Date'])
sp = sp[(sp.cls == 'promo_after_off') & sp.cls_prev.str.startswith('off')
        & sp.cls_next.str.startswith('off')].copy()
for v in ['Sales', 'Customers']:
    sp[f'{v}_ctrl'] = (sp[f'{v}_prev'] + sp[f'{v}_next']) / 2

sat_raw = lift(sp)
sat_corr = (1 + sat_raw) * (1 + eff) - 1 if stockpiling else sat_raw
print(f"\nPromo-week Saturdays matched: {len(sp):,}")
print(f"Saturday effect: raw {sat_raw:+.1%} (traffic {lift(sp, 'Customers'):+.1%}) → corrected {sat_corr:+.1%}")
print(sp.groupby('Cluster')[sp.columns].apply(lift).map('{:+.1%}'.format).rename('sat_effect'))

## 11. Save summary

In [ ]:
summary = {k: (None if pd.isna(v) else round(float(v), 4)) for k, v in {
    'matched_lift': overall.sales_lift,
    'traffic_lift': overall.traffic_lift,
    'basket_lift': overall.basket_lift,
    'stockpiling_gap': eff, 'stockpiling_ci_low': lo, 'stockpiling_ci_high': hi,
    'corrected_lift': corrected,
    'saturday_effect': sat_corr,
}.items()}
summary['stockpiling_significant'] = stockpiling
pd.Series(summary).to_json('../data/processed/promo_lift_summary.json', indent=2)
print(summary)